# ETL — Minnie65: DataSet & DataItem

Writes one `DataSet` record (`dataset_id = "minnie65_v1300_nuclei"`, `project_id = "minnie65"`) and one `DataItem` per nucleus from the CAVE `nucleus_detection_lookup_v1` view at materialization version 1300, plus the corresponding `DataItemDataSetAssociation` links. Cohort DataSets (e.g. `minnie65_v1300_csm_cluster`) and cell features are written by later notebooks.

In [1]:
import os

import caveclient
import pandas as pd
import polars as pl
import pyarrow as pa

from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Modality,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.2) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [2]:
OUTPUT_ROOT = output_root()
PROJECT_ID  = "minnie65"
DATASET_ID  = "minnie65_v1300_nuclei"
CAVE_DATASTACK   = "minnie65_phase3_v1"
CAVE_VERSION     = 1300
CAVE_VIEW        = "nucleus_detection_lookup_v1"

print(f"OUTPUT_ROOT      : {OUTPUT_ROOT}")
print(f"PROJECT_ID       : {PROJECT_ID}")
print(f"DATASET_ID       : {DATASET_ID}")
print(f"CAVE_DATASTACK   : {CAVE_DATASTACK}")
print(f"CAVE_VERSION     : {CAVE_VERSION}")
print(f"CAVE_VIEW        : {CAVE_VIEW}")

OUTPUT_ROOT      : ../scratch/em_patchseq_wnm_v3/
PROJECT_ID       : minnie65
DATASET_ID       : minnie65_v1300_nuclei
CAVE_DATASTACK   : minnie65_phase3_v1
CAVE_VERSION     : 1300
CAVE_VIEW        : nucleus_detection_lookup_v1


## Query CAVE

In [3]:
client = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CUSTOM_KEY"])
client.materialize.version = CAVE_VERSION

nuc_df = client.materialize.query_view(CAVE_VIEW)
nuc_df = nuc_df.query("pt_root_id != 0")

print("Shape:", nuc_df.shape)
nuc_df.head(3)

HTTPError: 503 Server Error: Service Temporarily Unavailable for url: https://minnie.microns-daf.com/materialize/version content:b'<html>\r\n<head><title>503 Service Temporarily Unavailable</title></head>\r\n<body>\r\n<center><h1>503 Service Temporarily Unavailable</h1></center>\r\n<hr><center>nginx</center>\r\n</body>\r\n</html>\r\n'

## Write `DataSet`

In [ ]:
dataset = DataSet(
    id=DATASET_ID,
    name="Minnie65 v1300 nucleus catalog",
    publication="doi.org/10.1038/s41586-025-08778-6",
    modality=Modality.ELECTRON_MICROSCOPY.value,
    project_id=PROJECT_ID,
)
result = write_models([dataset], output_root=OUTPUT_ROOT)
print(f"DataSet written: {result.rows_written} rows")

In [5]:
# Verification
ds_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataset/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == DATASET_ID))
    .filter(pl.col("id") == DATASET_ID)
)
print(ds_verify.shape)
print(ds_verify.head())
assert ds_verify.shape[0] == 1, f"Expected 1 DataSet row, got {ds_verify.shape[0]}"
assert ds_verify["id"][0] == DATASET_ID, "DataSet id mismatch"

(1, 5)
shape: (1, 5)
┌──────────────────────┬─────────────────┬──────────────────────┬─────────────────────┬────────────┐
│ id                   ┆ name            ┆ publication          ┆ modality            ┆ project_id │
│ ---                  ┆ ---             ┆ ---                  ┆ ---                 ┆ ---        │
│ str                  ┆ str             ┆ str                  ┆ str                 ┆ str        │
╞══════════════════════╪═════════════════╪══════════════════════╪═════════════════════╪════════════╡
│ minnie65_v1300_nucle ┆ Minnie65 v1300  ┆ doi.org/10.1038/s415 ┆ ELECTRON_MICROSCOPY ┆ minnie65   │
│ i                    ┆ nucleus catalog ┆ 86-025-087…          ┆                     ┆            │
└──────────────────────┴─────────────────┴──────────────────────┴─────────────────────┴────────────┘


## Write `DataItem`

In [6]:
dataitems = [
    DataItem(id=str(row.id), name=str(row.pt_root_id), project_id=PROJECT_ID)
    for row in nuc_df.itertuples()
]
n_appended = write_models(dataitems, output_root=OUTPUT_ROOT).rows_written
print(f"DataItem rows appended: {n_appended} (total in batch: {len(dataitems)})")

DataItem rows appended: 133969 (total in batch: 133969)


In [7]:
# Verification
di_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(di_verify.shape)
print(di_verify.head())
registered_ids = set(di_verify["id"].to_list())
dataitems_ids = [item.id for item in dataitems]
assert all(cid in registered_ids for cid in dataitems_ids), "Some DataItem ids are missing from the table"
assert di_verify["id"].n_unique() == di_verify.shape[0], "Duplicate DataItem ids detected"

(133969, 4)
shape: (5, 4)
┌────────┬────────────────────┬───────────────────┬────────────┐
│ id     ┆ name               ┆ neuroglancer_link ┆ project_id │
│ ---    ┆ ---                ┆ ---               ┆ ---        │
│ str    ┆ str                ┆ str               ┆ str        │
╞════════╪════════════════════╪═══════════════════╪════════════╡
│ 373879 ┆ 864691136090135607 ┆ null              ┆ minnie65   │
│ 201858 ┆ 864691135373893678 ┆ null              ┆ minnie65   │
│ 600774 ┆ 864691135682378744 ┆ null              ┆ minnie65   │
│ 408486 ┆ 864691135194387242 ┆ null              ┆ minnie65   │
│ 598774 ┆ 864691135741608653 ┆ null              ┆ minnie65   │
└────────┴────────────────────┴───────────────────┴────────────┘


## Write `DataItemDataSetAssociation`

In [8]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=item.id,
        dataset_id=DATASET_ID,
        project_id=PROJECT_ID,
    )
    for item in dataitems
]
result = write_models(associations, output_root=OUTPUT_ROOT)
print(f"DataItemDataSetAssociation written: {result.rows_written} rows")

DataItemDataSetAssociation written: 133969 rows


In [9]:
# Verification
assoc_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter(pl.col("project_id") == PROJECT_ID)
    .filter(pl.col("dataset_id") == DATASET_ID)
)
print(assoc_verify.shape)
print(assoc_verify.head())
assert assoc_verify.shape[0] == len(associations), (
    f"Expected {len(associations)} association rows, got {assoc_verify.shape[0]}"
)
assert (assoc_verify["dataset_id"] == DATASET_ID).all(), "Not all associations point to DATASET_ID"

(133969, 3)
shape: (5, 3)
┌─────────────┬───────────────────────┬────────────┐
│ dataitem_id ┆ dataset_id            ┆ project_id │
│ ---         ┆ ---                   ┆ ---        │
│ str         ┆ str                   ┆ str        │
╞═════════════╪═══════════════════════╪════════════╡
│ 373879      ┆ minnie65_v1300_nuclei ┆ minnie65   │
│ 201858      ┆ minnie65_v1300_nuclei ┆ minnie65   │
│ 600774      ┆ minnie65_v1300_nuclei ┆ minnie65   │
│ 408486      ┆ minnie65_v1300_nuclei ┆ minnie65   │
│ 598774      ┆ minnie65_v1300_nuclei ┆ minnie65   │
└─────────────┴───────────────────────┴────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataset/` | `DataSet` | 1 |
| `dataitem/` | `DataItem` | `len(nuc_df)` |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | `len(nuc_df)` |

**Intentionally not written here:**
- Cohort DataSets (e.g. `minnie65_v1300_csm_cluster`) — each cohort is an additional `DataSet` row plus `DataItemDataSetAssociation` rows pointing at the same `DataItem` ids; written by `_02`/`_03` notebooks.
- Cell features (`pt_position`, cell type labels, etc.) — written in `_02` as `CellFeature` records.